In [69]:
import pandas as pd

# Loads every sheet into a dict: {"Sheet1": df1, "Sheet2": df2, ...}
all_sheets = pd.read_excel("DV.xlsx", sheet_name=None)
for name, df in all_sheets.items():
    df.to_csv(f"{name}.csv", index=False)

In [70]:
import numpy as np
import re

df_emission = pd.read_csv("Emission.csv", header=None)

print(f"Kích thước dữ liệu gốc: {df_emission.shape}")
display(df_emission.head())

Kích thước dữ liệu gốc: (55, 116)


,0,1,2,3,4,5,6,7,8,9,...,106,107,108,109,110,111,112,113,114,115
0,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 106,Unnamed: 107,Unnamed: 108,Unnamed: 109,Unnamed: 110,Unnamed: 111,Unnamed: 112,Unnamed: 113,Unnamed: 114,Unnamed: 115
1,Dust/ Bụi,NaN,NaN,Jan,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,YTD,Q1,Q2,Q3,Q4
2,Stack No.,Process,Diameter/ Đường kính ống khói,Sampling \n(Month/Year),Stack \nTemperature,%Oxygen,Gas Velocity,Flow rate/ Lưu lượng,Dust Emission\nActual O2/ Kết quả đo bụi,Dust Emission \nrate/ Tỉ lệ bụi,...,Flow rate,Dust Emission\nActual O2,Dust Emission \nrate,Operation \nHour,Dust Emission,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,m,MM/YYYY,๐C,%,m/s,m3/min,mg/m3,kg/hr,...,m3/min,mg/m3,kg/hr,hr,kg,NaN,NaN,NaN,NaN,NaN
4,Sấy phun 1,NaN,1.3,2/12/2019,84.3,NaN,NaN,1416.666667,41,3.485,...,1416.666667,43.4,3.689,660,2434.74,NaN,NaN,NaN,NaN,NaN


In [71]:
# Tìm index của các dòng chứa chữ 'Stack No.'
stack_no_rows = df_emission[df_emission[0].astype(str).str.strip().str.lower() == 'stack no.'].index.tolist()
print("Vị trí các dòng 'Stack No.' tìm thấy:", stack_no_rows)

#Tiến hành cắt bảng
tables = {}
for i in range(len(stack_no_rows)):
    # Dòng bắt đầu của bảng là dòng NGAY TRÊN dòng 'Stack No.'
    start_idx = stack_no_rows[i] - 1
    # Lấy tên bảng
    table_name = str(df_emission.iloc[start_idx, 0]).strip()
    end_idx = stack_no_rows[i+1] - 1 if i + 1 < len(stack_no_rows) else len(df_emission)
    df_sub = df_emission.iloc[start_idx:end_idx].reset_index(drop=True)

    # Dọn dẹp các dòng trống rác cuối bảng
    df_sub = df_sub.dropna(how='all')

    tables[table_name] = df_sub

print(f"\n Đã tách thành công {len(tables)} bảng:")
for name, df in tables.items():
    print(f" - Bảng '{name}': {df.shape[0]} dòng, {df.shape[1]} cột")

Vị trí các dòng 'Stack No.' tìm thấy: [2, 20, 38]

 Đã tách thành công 3 bảng:
 - Bảng 'Dust/ Bụi': 17 dòng, 116 cột
 - Bảng 'Sox/ Khí Sox': 17 dòng, 116 cột
 - Bảng 'Nox': 18 dòng, 116 cột


In [72]:
# Từ điển ánh xạ cụm từ chỉ định
translation_map = {
    "Sấy phun 1": "Spray Dryer 1",
    "Sấy phun 2": "Spray Dryer 2",
    "Ống khói lò nung xương 1,2": "Biscuit Kiln Exhaust Stack 1, 2",
    "Ống khói lò nung xương 3,4": "Biscuit Kiln Exhaust Stack 3, 4",
    "Ống khói lò nung xương 5,6": "Biscuit Kiln Exhaust Stack 5, 6",
    "Ống khói lò nung men số 1": "Glaze Kiln Exhaust Stack No. 1",
    "Ống khói lò nung men số 2": "Glaze Kiln Exhaust Stack No. 2",
    "Ống khói lò nung men số 3": "Glaze Kiln Exhaust Stack No. 3",
    "Ống khói lò nung men số 4": "Glaze Kiln Exhaust Stack No. 4",
    "Ống khói lò nung men số 5": "Glaze Kiln Exhaust Stack No. 5",
    "Ống khói lò nung men số 6": "Glaze Kiln Exhaust Stack No. 6",
    "Sampling \n(Month/Year)": "Sampling"
    }

viet_chars = (
    r'àáâãèéêìíòóôõùúýăđơư'
    r'ạảấầẩẫậắằẳẵặẹẻẽếềểễệ'
    r'ỉịọỏốồổỗộớờởỡợụủứừửữự'
    r'ỳỵỷỹÀÁÂÃÈÉÊÌÍÒÓÔÕÙÚÝĂĐƠƯ'
    r'ẠẢẤẦẨẪẬẮẰẲẴẶẸẺẼẾỀỂỄỆ'
    r'ỈỊỌỎỐỒỔỖỘỚỜỞỠỢỤỦỨỪỬỮỰ'
    r'ỲỴỶỸ'
)

def process_and_remove_vietnamese(text):
    if not isinstance(text, str) or pd.isna(text):
        return text

    # Thay thế các cụm từ chỉ định trước
    for vi, en in translation_map.items():
        if vi in text:
            text = text.replace(vi, en)

    # Xóa phần trong ngoặc chứa tiếng Việt
    text = re.sub(rf'\s*\([^)]*[{viet_chars}][^)]*\)', '', text)
    # Xóa phần sau dấu gạch chéo '/' chứa tiếng Việt
    text = re.sub(rf'\s*/[^/]*[{viet_chars}].*', '', text)
    # Xóa các cụm từ tiếng Việt đứng trơ trọi ở cuối chuỗi
    text = re.sub(rf'\s+[{viet_chars}a-zA-Z]*[{viet_chars}][\w\s]*$', '', text)
    # Xóa khoảng trắng thừa
    text = re.sub(r'\s+', ' ', text).strip()
    return text

processed_tables = {}

for old_name, df in tables.items():
    # Đổi tên bảng
    new_table_name = process_and_remove_vietnamese(old_name)

    # Xử lý dữ liệu bên trong bảng
    new_df = df.copy()
    string_cols = new_df.select_dtypes(include=['object', 'string']).columns
    for col in string_cols:
        new_df[col] = new_df[col].apply(process_and_remove_vietnamese)

    processed_tables[new_table_name] = new_df

    # In kết quả 50 dòng đầu
    print(f"\n--- Bảng cũ: '{old_name}' -> Tên mới: '{new_table_name}' ---")
    display(new_df.head(50))

# Cập nhật lại biến tables gốc
tables = processed_tables


--- Bảng cũ: 'Dust/ Bụi' -> Tên mới: 'Dust' ---


,0,1,2,3,4,5,6,7,8,9,...,106,107,108,109,110,111,112,113,114,115
0,Dust,NaN,NaN,Jan,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,YTD,Q1,Q2,Q3,Q4
1,Stack No.,Process,Diameter,Sampling,Stack Temperature,%Oxygen,Gas Velocity,Flow rate,Dust Emission Actual O2,Dust Emission rate,...,Flow rate,Dust Emission Actual O2,Dust Emission rate,Operation Hour,Dust Emission,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,m,MM/YYYY,๐C,%,m/s,m3/min,mg/m3,kg/hr,...,m3/min,mg/m3,kg/hr,hr,kg,NaN,NaN,NaN,NaN,NaN
3,Spray Dryer 1,NaN,1.3,2/12/2019,84.3,NaN,NaN,1416.666667,41,3.485,...,1416.666667,43.4,3.689,660,2434.74,NaN,NaN,NaN,NaN,NaN
4,Spray Dryer 2,NaN,1.3,2/12/2019,87.1,NaN,NaN,1416.666667,43,3.655,...,1416.666667,42.3,3.5955,436,1567.638,NaN,NaN,NaN,NaN,NaN
5,"Biscuit Kiln Exhaust Stack 1, 2",NaN,0.8,2/12/2019,153.2,NaN,NaN,173.3333333,52,0.5408,...,154,45.1,0.416724,744,310.042656,NaN,NaN,NaN,NaN,NaN
6,"Biscuit Kiln Exhaust Stack 3, 4",NaN,0.8,2/12/2019,159.1,NaN,NaN,188.3333333,53,0.5989,...,158.3333333,42.6,0.4047,744,301.0968,NaN,NaN,NaN,NaN,NaN
7,"Biscuit Kiln Exhaust Stack 5, 6",NaN,0.8,2/12/2019,146.5,NaN,NaN,175.5,50,0.5265,...,194.1666667,43.9,0.511435,744,380.50764,NaN,NaN,NaN,NaN,NaN
8,Glaze Kiln Exhaust Stack No. 1,NaN,0.8,29/9/2019,157.3,NaN,NaN,168.6666667,51,0.51612,...,178.3333333,43.7,0.46759,744,347.88696,NaN,NaN,NaN,NaN,NaN
9,Glaze Kiln Exhaust Stack No. 2,NaN,0.8,29/9/2019,124.3,NaN,NaN,141.5,52,0.44148,...,155.6666667,44.2,0.412828,744,307.144032,NaN,NaN,NaN,NaN,NaN



--- Bảng cũ: 'Sox/ Khí Sox' -> Tên mới: 'Sox' ---


,0,1,2,3,4,5,6,7,8,9,...,106,107,108,109,110,111,112,113,114,115
0,Sox,NaN,NaN,Jan,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,YTD,Q1,Q2,Q3,Q4
1,Stack No.,Process,Diameter,Sampling,Stack Temperature,%Oxygen,Gas Velocity,Flow rate,SOx Emission Actual O2,SOx Emission rate,...,Flow rate,SOx Emission Actual O2,SOx Emission rate,Operation Hour,SOx Emission,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,m,MM/YYYY,๐C,%,m/s,m3/min,mg/m3,kg/hr,...,m3/min,mg/m3,kg/hr,hr,kg,NaN,NaN,NaN,NaN,NaN
3,Spray Dryer 1,NaN,1.3,2/12/2019,84.3,NaN,NaN,1416.666667,15.72,1.3362,...,1416.666667,94.2,8.007,660,5284.62,NaN,NaN,NaN,NaN,NaN
4,Spray Dryer 2,NaN,1.3,2/12/2019,87.1,NaN,NaN,1416.666667,18.34,1.5589,...,1416.666667,94.7,8.0495,436,3509.582,NaN,NaN,NaN,NaN,NaN
5,"Biscuit Kiln Exhaust Stack 1, 2",NaN,0.8,2/12/2019,153.2,NaN,NaN,173.3333333,83.84,0.871936,...,154,92.4,0.853776,744,635.209344,NaN,NaN,NaN,NaN,NaN
6,"Biscuit Kiln Exhaust Stack 3, 4",NaN,0.8,2/12/2019,159.1,NaN,NaN,188.3333333,84.71,0.957223,...,158.3333333,96.1,0.91295,744,679.2348,NaN,NaN,NaN,NaN,NaN
7,"Biscuit Kiln Exhaust Stack 5, 6",NaN,0.8,2/12/2019,146.5,NaN,NaN,175.5,80.19,0.8444007,...,194.1666667,93.8,1.09277,744,813.02088,NaN,NaN,NaN,NaN,NaN
8,Glaze Kiln Exhaust Stack No. 1,NaN,0.8,29/9/2019,157.3,NaN,NaN,168.6666667,6.38,0.0645656,...,178.3333333,96.2,1.02934,744,765.82896,NaN,NaN,NaN,NaN,NaN
9,Glaze Kiln Exhaust Stack No. 2,NaN,0.8,29/9/2019,124.3,NaN,NaN,141.5,6.42,0.0545058,...,155.6666667,97.8,0.913452,744,679.608288,NaN,NaN,NaN,NaN,NaN



--- Bảng cũ: 'Nox' -> Tên mới: 'Nox' ---


,0,1,2,3,4,5,6,7,8,9,...,106,107,108,109,110,111,112,113,114,115
0,Nox,NaN,NaN,Jan,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,YTD,Q1,Q2,Q3,Q4
1,Stack No.,Process,Diameter,Sampling,Stack Temperature,%Oxygen,Gas Velocity,Flow rate,NOx Emission Actual O2,NOx Emission rate,...,Flow rate,NOx Emission Actual O2,NOx Emission rate,Operation Hour,NOx Emission,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,m,MM/YYYY,๐C,%,m/s,m3/min,mg/m3,kg/hr,...,m3/min,mg/m3,kg/hr,hr,kg,NaN,NaN,NaN,NaN,NaN
3,Spray Dryer 1,NaN,1.3,2/12/2019,84.3,NaN,NaN,1416.666667,4.4,0.374,...,1416.666667,71.6,6.086,660,4016.76,NaN,NaN,NaN,NaN,NaN
4,Spray Dryer 2,NaN,1.3,2/12/2019,87.1,NaN,NaN,1416.666667,4.6,0.391,...,1416.666667,67.9,5.7715,436,2516.374,NaN,NaN,NaN,NaN,NaN
5,"Biscuit Kiln Exhaust Stack 1, 2",NaN,0.8,2/12/2019,153.2,NaN,NaN,173.3333333,5.1,0.05304,...,154,72.7,0.671748,744,499.780512,NaN,NaN,NaN,NaN,NaN
6,"Biscuit Kiln Exhaust Stack 3, 4",NaN,0.8,2/12/2019,159.1,NaN,NaN,188.3333333,4.6,0.05198,...,158.3333333,68.9,0.65455,744,486.9852,NaN,NaN,NaN,NaN,NaN
7,"Biscuit Kiln Exhaust Stack 5, 6",NaN,0.8,2/12/2019,146.5,NaN,NaN,175.5,4.8,0.050544,...,194.1666667,70.4,0.82016,744,610.19904,NaN,NaN,NaN,NaN,NaN
8,Glaze Kiln Exhaust Stack No. 1,NaN,0.8,29/9/2019,157.3,NaN,NaN,168.6666667,5.7,0.057684,...,178.3333333,72.4,0.77468,744,576.36192,NaN,NaN,NaN,NaN,NaN
9,Glaze Kiln Exhaust Stack No. 2,NaN,0.8,29/9/2019,124.3,NaN,NaN,141.5,5.5,0.046695,...,155.6666667,70.7,0.660338,744,491.291472,NaN,NaN,NaN,NaN,NaN


In [73]:
 def transform_to_long_format(df):
    # 1. Tìm vị trí dòng chứa tên các biến
    header_idx = -1
    for i in range(min(10, len(df))):
        # Nối các ô trong dòng thành chuỗi để tìm kiếm
        row_str = " ".join([str(x) for x in df.iloc[i].values])
        if "Stack No." in row_str or "Stack No" in row_str:
            header_idx = i
            break

    if header_idx == -1:
        # Nếu không tìm thấy cấu trúc bảng chuẩn, trả về bảng gốc
        return df

    # Cắt các dòng tiêu đề ra để xử lý
    month_row = df.iloc[header_idx - 1].fillna("").astype(str)
    var_row = df.iloc[header_idx].fillna("").astype(str)
    unit_row = df.iloc[header_idx + 1].fillna("").astype(str)

    # Quét dòng tháng (Month) và Forward Fill (Điền tới)
    # Ví dụ: [Jan, rỗng, rỗng, Feb, rỗng] -> [Jan, Jan, Jan, Feb, Feb]
    months = []
    curr_month = ""
    for val in month_row:
        val_clean = val.strip()
        # Cập nhật tháng nếu ô có chữ và không phải tên chất (như Nox, SOx...)
        if val_clean and val_clean.lower() not in ['nox', 'sox', 'co', 'pm', 'dust', 'nan']:
            curr_month = val_clean
        months.append(curr_month)
    # Tìm tháng hợp lệ đầu tiên (vd: 'Jan')
    first_valid_month = next((m for m in months if m != ""), "")

    # Lấp ngược chữ 'Jan' vào các vị trí trống ở đầu mảng
    if first_valid_month:
        for i in range(len(months)):
            if months[i] == "":
                months[i] = first_valid_month
    # Tìm vị trí cột "Stack No."
    try:
        stack_col_idx = list(var_row).index("Stack No.")
    except ValueError:
        return df

    # 4. Duyệt qua từng dòng dữ liệu và biến đổi từ ngang sang dọc
    records = []
    for idx in range(header_idx + 2, len(df)):
        row = df.iloc[idx]
        stack_val = row.iloc[stack_col_idx]

        # Bỏ qua nếu dòng này không có tên ống khói (dòng trống)
        if pd.isna(stack_val) or str(stack_val).strip() == "" or str(stack_val).strip() == "nan":
            continue

        # Duyệt qua từng cột của dòng đó
        for col_idx in range(len(var_row)):
            var_name = var_row.iloc[col_idx].strip()

            # BỎ QUA các cột không hợp lệ
            if var_name in ["Process", "Stack No.", "", "nan"]:
                continue

            period = months[col_idx]
            unit = unit_row.iloc[col_idx].strip()
            if unit == "nan": unit = ""

            value = row.iloc[col_idx]

            # ĐIỀN 0 CHO Ô TRỐNG
            if pd.isna(value) or str(value).strip() in ["", "nan"]:
                value = 0

            # Lưu lại thành một dòng dữ liệu mới
            records.append({
                "Stack No.": stack_val,
                "period": period,
                "variable": var_name,
                "unit": unit,
                "value": value
            })

    return pd.DataFrame(records)

# ÁP DỤNG CHO TẤT CẢ CÁC BẢNG

final_tables = {}

print("\nĐang tiến hành chuyển đổi cấu trúc bảng")

for name, df in processed_tables.items():

    long_df = transform_to_long_format(df)

    final_tables[name] = long_df

    print(f"\n--- Bảng: '{name}' sau khi chuyển đổi ---")
    display(long_df.head(50))


Đang tiến hành chuyển đổi cấu trúc bảng

--- Bảng: 'Dust' sau khi chuyển đổi ---


,Stack No.,period,variable,unit,value
0,Spray Dryer 1,Jan,Diameter,m,1.3
1,Spray Dryer 1,Jan,Sampling,MM/YYYY,2/12/2019
2,Spray Dryer 1,Jan,Stack Temperature,๐C,84.3
3,Spray Dryer 1,Jan,%Oxygen,%,0
4,Spray Dryer 1,Jan,Gas Velocity,m/s,0
5,Spray Dryer 1,Jan,Flow rate,m3/min,1416.666667
6,Spray Dryer 1,Jan,Dust Emission Actual O2,mg/m3,41
7,Spray Dryer 1,Jan,Dust Emission rate,kg/hr,3.485
8,Spray Dryer 1,Jan,Operation Hour,hr,452
9,Spray Dryer 1,Jan,Dust Emission,kg,1575.22



--- Bảng: 'Sox' sau khi chuyển đổi ---


,Stack No.,period,variable,unit,value
0,Spray Dryer 1,Jan,Diameter,m,1.3
1,Spray Dryer 1,Jan,Sampling,MM/YYYY,2/12/2019
2,Spray Dryer 1,Jan,Stack Temperature,๐C,84.3
3,Spray Dryer 1,Jan,%Oxygen,%,0
4,Spray Dryer 1,Jan,Gas Velocity,m/s,0
5,Spray Dryer 1,Jan,Flow rate,m3/min,1416.666667
6,Spray Dryer 1,Jan,SOx Emission Actual O2,mg/m3,15.72
7,Spray Dryer 1,Jan,SOx Emission rate,kg/hr,1.3362
8,Spray Dryer 1,Jan,Operation Hour,hr,452
9,Spray Dryer 1,Jan,SOx Emission,kg,603.9624



--- Bảng: 'Nox' sau khi chuyển đổi ---


,Stack No.,period,variable,unit,value
0,Spray Dryer 1,Jan,Diameter,m,1.3
1,Spray Dryer 1,Jan,Sampling,MM/YYYY,2/12/2019
2,Spray Dryer 1,Jan,Stack Temperature,๐C,84.3
3,Spray Dryer 1,Jan,%Oxygen,%,0
4,Spray Dryer 1,Jan,Gas Velocity,m/s,0
5,Spray Dryer 1,Jan,Flow rate,m3/min,1416.666667
6,Spray Dryer 1,Jan,NOx Emission Actual O2,mg/m3,4.4
7,Spray Dryer 1,Jan,NOx Emission rate,kg/hr,0.374
8,Spray Dryer 1,Jan,Operation Hour,hr,452
9,Spray Dryer 1,Jan,NOx Emission,kg,169.048


In [74]:
print("\nĐang tiến hành gộp tất cả các bảng và tách cột Sampling...")

# Gộp tất cả các bảng lại trước
list_of_dfs_to_combine = []
for table_name, df in final_tables.items():
    temp_df = df.copy()
    temp_df.insert(0, 'category', table_name)
    list_of_dfs_to_combine.append(temp_df)

if list_of_dfs_to_combine:
    final_combined_df = pd.concat(list_of_dfs_to_combine, ignore_index=True)

    # 2. Tách các dòng chứa biến 'Sampling' ra để lấy ngày
    sampling_mask = final_combined_df['variable'].str.strip() == 'Sampling'
    sampling_df = final_combined_df[sampling_mask].copy()

    sampling_dates = sampling_df[['category', 'Stack No.', 'period', 'value']].rename(columns={'value': 'Sampling'})

    # Loại bỏ các dòng 'Sampling' ra khỏi dữ liệu dọc chính
    main_df = final_combined_df[~sampling_mask].copy()

    # Gộp ngày (Merge) vào lại dữ liệu chính dựa trên 3 khóa
    final_df = pd.merge(main_df, sampling_dates, on=['category', 'Stack No.', 'period'], how='left')

    # Ép cột Sampling về chuẩn Datetime
    final_df['Sampling'] = pd.to_datetime(final_df['Sampling'], dayfirst=True, errors='coerce')

    # Ép cột value về dạng float
    final_df['value'] = pd.to_numeric(final_df['value'], errors='coerce').fillna(0).astype(float)

    # Sắp xếp lại thứ tự cột
    cols = ['category', 'Stack No.', 'period', 'variable', 'unit', 'Sampling', 'value']
    final_df = final_df[cols]

    # Xuất ra file CSV cuối cùng
    final_output_file = "Emission_cleaned.csv"
    final_df.to_csv(final_output_file, index=False, encoding='utf-8-sig')

    # In thông báo và kết quả ra màn hình
    print(f"-> Thông tin dữ liệu:")
    final_df.info()
    print(f"-> File kết quả cuối cùng đã được lưu tên: '{final_output_file}'")
    print("\n--- XEM THỬ 20 DÒNG ĐẦU CỦA BẢNG TỔNG ---")
    display(final_df.head(20))
else:
    print("Lỗi: Không tìm thấy bảng nào để gộp!")


Đang tiến hành gộp tất cả các bảng và tách cột Sampling...
-> Thông tin dữ liệu:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3783 entries, 0 to 3782
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   category   3783 non-null   object        
 1   Stack No.  3783 non-null   object        
 2   period     3783 non-null   object        
 3   variable   3783 non-null   object        
 4   unit       3783 non-null   object        
 5   Sampling   3201 non-null   datetime64[ns]
 6   value      3783 non-null   float64       
dtypes: datetime64[ns](1), float64(1), object(5)
memory usage: 207.0+ KB
-> File kết quả cuối cùng đã được lưu tên: 'Emission_cleaned.csv'

--- XEM THỬ 20 DÒNG ĐẦU CỦA BẢNG TỔNG ---


,category,Stack No.,period,variable,unit,Sampling,value
0,Dust,Spray Dryer 1,Jan,Diameter,m,2019-12-02,1.300000
1,Dust,Spray Dryer 1,Jan,Stack Temperature,๐C,2019-12-02,84.300000
2,Dust,Spray Dryer 1,Jan,%Oxygen,%,2019-12-02,0.000000
3,Dust,Spray Dryer 1,Jan,Gas Velocity,m/s,2019-12-02,0.000000
4,Dust,Spray Dryer 1,Jan,Flow rate,m3/min,2019-12-02,1416.666667
5,Dust,Spray Dryer 1,Jan,Dust Emission Actual O2,mg/m3,2019-12-02,41.000000
6,Dust,Spray Dryer 1,Jan,Dust Emission rate,kg/hr,2019-12-02,3.485000
7,Dust,Spray Dryer 1,Jan,Operation Hour,hr,2019-12-02,452.000000
8,Dust,Spray Dryer 1,Jan,Dust Emission,kg,2019-12-02,1575.220000
9,Dust,Spray Dryer 1,Feb,Stack Temperature,๐C,2019-12-02,84.300000
